# run_convo_asd

Runs the ASD / video-sync pipeline (`video-sync-nbu-main`) for the conversation-task
recordings of `anilu_comparison` patients.

**One SLURM job is submitted per camera** (discovered from `{pt}Datafile/VIDEO/`).
Each job calls `cli_emu_time` with `--cam-serial <serial>`, restricting it to one camera.
Success is detected by the presence of a synced MP4 in the expected output path.

**LR-ASD** (section 6+): one GPU job per synced MP4, also one per camera.

**Only patients ≥ YFB** are processed (video data not set up for earlier patients).

Worker: [convo_asd_worker.py](convo_asd_worker.py)

In [1]:
from pathlib import Path
import subprocess
import re
import pandas as pd

In [2]:
# ── Paths ────────────────────────────────────────────────────────────────────
ANILU_ROOT    = Path("/mnt/labworlds/Hayden/Hayden_Lab/speech_247/anilu_comparison")
STITCHED_ROOT = Path("/mnt/stitched/EMU-18112")         # {pt}/ contains NSP task subdirs
DATALAKE_ROOT = Path("/mnt/datalake/data/emu")          # {pt}Datafile/VIDEO/ camera segments
REPO_ROOT     = Path("/scratch/tahaismail424/video-sync-nbu-main")
WORKER_PATH   = Path("/scratch/tahaismail424/speech_247/notebooks/REBIRTH_2!/convo_comparison/convo_asd_worker.py")
PYTHON_BIN    = "/scratch/tahaismail424/miniforge3/envs/video_sync_db/bin/python3"
CONDA_ACTIVATE = "source /scratch/tahaismail424/.bash_profile && conda activate video_sync_db"

# ── SLURM settings ──────────────────────────────────────────────────────────
PARTITION  = "Universal"
CPUS       = 8
MEM        = "64G"      # convo recordings are long; generous budget
TIME       = "24:00:00"
QOS        = "big_batch_tier"
LOG_LEVEL  = "INFO"

# ── Patient filter ──────────────────────────────────────────────────────────
# Only patients >= YFB have video data set up.
MIN_PATIENT = "YFB"

# Set True to resubmit even patients that already have _SUCCESS
FORCE_RESUBMIT = False

# Set to e.g. 1 for a quick single-patient test run
MAX_PATIENTS: int | None = None

SLURM_LOGS_ROOT    = ANILU_ROOT.parent.parent / "convo_comparison" / "asd_slurm_logs"
SLURM_SCRIPTS_ROOT = ANILU_ROOT.parent.parent / "convo_comparison" / "asd_slurm_scripts"
SLURM_LOGS_ROOT.mkdir(parents=True, exist_ok=True)
SLURM_SCRIPTS_ROOT.mkdir(parents=True, exist_ok=True)

print("ANILU_ROOT:   ", ANILU_ROOT)
print("STITCHED_ROOT:", STITCHED_ROOT)
print("DATALAKE_ROOT:", DATALAKE_ROOT)
print("REPO_ROOT:    ", REPO_ROOT)
print("WORKER:       ", WORKER_PATH)
print("LOGS:         ", SLURM_LOGS_ROOT)

ANILU_ROOT:    /mnt/labworlds/Hayden/Hayden_Lab/speech_247/anilu_comparison
STITCHED_ROOT: /mnt/stitched/EMU-18112
DATALAKE_ROOT: /mnt/datalake/data/emu
REPO_ROOT:     /scratch/tahaismail424/video-sync-nbu-main
WORKER:        /scratch/tahaismail424/speech_247/notebooks/REBIRTH_2!/convo_comparison/convo_asd_worker.py
LOGS:          /mnt/labworlds/Hayden/Hayden_Lab/convo_comparison/asd_slurm_logs


## 1 — Discover patients & cameras

In [3]:
def discover_cameras() -> list[dict]:
    """
    For each anilu_comparison patient >= MIN_PATIENT, enumerate cameras
    from {pt}Datafile/VIDEO/ and return one row per (patient, cam_serial).

    Camera serials are extracted from MP4 filenames inside the date subdirs:
      VIDEO/{date}/{pt}_{date}_{time}.{cam_serial}.mp4

    Success is detected by the presence of a synced MP4 in the expected
    cli_emu_time output path: out_dir/{pt}/{task_keyword}/{cam_serial}/synced_video/*.mp4
    """
    rows = []
    for pt_dir in sorted(ANILU_ROOT.iterdir()):
        if not pt_dir.is_dir():
            continue
        pt = pt_dir.name
        if not re.match(r"Y[A-Z]{2}$", pt):
            continue
        if pt < MIN_PATIENT:
            continue

        # Task keyword from nsp/ subdir
        nsp_dir = pt_dir / "nsp"
        nsp_tasks = [d.name for d in sorted(nsp_dir.iterdir()) if d.is_dir()] if nsp_dir.exists() else []
        task_keyword = nsp_tasks[0] if nsp_tasks else None

        # Stitched NSP dir
        stitched_pt_dir = STITCHED_ROOT / pt
        task_in_stitched = False
        if stitched_pt_dir.exists() and task_keyword:
            task_dir = stitched_pt_dir / task_keyword
            task_in_stitched = (
                task_dir.exists()
                and any(task_dir.glob("*NSP-1.nev"))
                and any(task_dir.glob("*NSP-1.ns5"))
            )

        # Enumerate camera serials from VIDEO/{date}/{pt}_{date}_{time}.{serial}.mp4
        video_dir = DATALAKE_ROOT / f"{pt}Datafile" / "VIDEO"
        cameras = sorted({
            f.stem.rsplit(".", 1)[-1]
            for date_dir in video_dir.iterdir() if date_dir.is_dir()
            for f in date_dir.glob("*.mp4")
        }) if video_dir.exists() else []

        out_dir = pt_dir / "video"
        pt_ready = bool(task_keyword and task_in_stitched and cameras)

        for cam_serial in cameras:
            synced_dir = out_dir / pt / task_keyword / cam_serial / "synced_video" if task_keyword else None
            has_mp4 = (
                synced_dir is not None
                and synced_dir.exists()
                and any(synced_dir.glob("*.mp4"))
            )
            ready = pt_ready
            submit = ready and (not has_mp4 or FORCE_RESUBMIT)

            rows.append({
                "patient":          pt,
                "task_keyword":     task_keyword,
                "cam_serial":       cam_serial,
                "stitched_pt_dir":  stitched_pt_dir,
                "video_dir":        video_dir,
                "out_dir":          out_dir,
                "synced_dir":       synced_dir,
                "task_in_stitched": task_in_stitched,
                "has_mp4":          has_mp4,
                "ready":            ready,
                "submit":           submit,
            })

    return rows


camera_rows = discover_cameras()
if MAX_PATIENTS is not None:
    pts_seen: list[str] = []
    filtered = []
    for r in camera_rows:
        if r["patient"] not in pts_seen:
            pts_seen.append(r["patient"])
        if pts_seen.index(r["patient"]) < MAX_PATIENTS:
            filtered.append(r)
    camera_rows = filtered

status_df = pd.DataFrame(camera_rows)
n_pts = status_df["patient"].nunique() if not status_df.empty else 0
print(f"Patients found:   {n_pts}")
print(f"Camera jobs:      {len(status_df)}")
if not status_df.empty:
    print(f"  task_in_stitched: {status_df['task_in_stitched'].sum()}")
    print(f"  has synced MP4:   {status_df['has_mp4'].sum()}")
    print(f"  ready:            {status_df['ready'].sum()}")
    print(f"  to submit:        {status_df['submit'].sum()}")
print()
status_df[["patient", "task_keyword", "cam_serial", "task_in_stitched", "has_mp4", "submit"]]

Patients found:   12
Camera jobs:      89
  task_in_stitched: 89
  has synced MP4:   79
  ready:            89
  to submit:        10



,patient,task_keyword,cam_serial,task_in_stitched,has_mp4,submit
0,YFC,EMU-0028_convo,18486634,True,True,False
1,YFC,EMU-0028_convo,18486638,True,True,False
2,YFC,EMU-0028_convo,23512014,True,True,False
3,YFC,EMU-0028_convo,23512906,True,True,False
4,YFC,EMU-0028_convo,23512908,True,False,True
...,...,...,...,...,...,...
84,YFV,EMU-0208_convo,23512012,True,True,False
85,YFV,EMU-0208_convo,23512013,True,True,False
86,YFV,EMU-0208_convo,23512014,True,True,False
87,YFV,EMU-0208_convo,23512906,True,True,False


## 2 — Submit SLURM jobs (one per camera)

In [4]:
submitted = []
failed    = []

to_submit = status_df[status_df["submit"]] if not status_df.empty else pd.DataFrame()
print(f"Submitting {len(to_submit)} jobs…")

for _, row in to_submit.iterrows():
    pt           = row["patient"]
    task_keyword = row["task_keyword"]
    video_dir    = row["video_dir"]
    out_dir      = row["out_dir"]
    cam_serial   = row["cam_serial"]
    job_name     = f"vsync_{pt}_{cam_serial}"
    log_stem     = f"{pt}_{cam_serial}"

    lines = [
        "#!/bin/bash",
        f"#SBATCH --job-name={job_name}",
        f"#SBATCH --partition={PARTITION}",
        f"#SBATCH --qos={QOS}",
        f"#SBATCH --cpus-per-task={CPUS}",
        f"#SBATCH --mem={MEM}",
        f"#SBATCH --time={TIME}",
        f"#SBATCH --output={SLURM_LOGS_ROOT}/{log_stem}_%j.out",
        f"#SBATCH --error={SLURM_LOGS_ROOT}/{log_stem}_%j.err",
        "",
        "set -eo pipefail",
        "",
        CONDA_ACTIVATE,
        "",
        'echo "HOSTNAME: $(hostname)"',
        'echo "START: $(date)"',
        f'echo "PATIENT:    {pt}"',
        f'echo "TASK:       {task_keyword}"',
        f'echo "CAM_SERIAL: {cam_serial}"',
        "",
        f"{PYTHON_BIN} {WORKER_PATH} \\",
        f"    {pt} \\",
        f"    {STITCHED_ROOT} \\",
        f"    {video_dir} \\",
        f"    {out_dir} \\",
        f"    {REPO_ROOT} \\",
        f"    --task-keyword {task_keyword} \\",
        f"    --cam-serial {cam_serial} \\",
        f"    --log-level {LOG_LEVEL}",
        "",
        'echo "END: $(date)"',
    ]
    sbatch_text = "\n".join(lines) + "\n"
    sbatch_path = SLURM_SCRIPTS_ROOT / f"{log_stem}_vsync.sbatch"
    sbatch_path.write_text(sbatch_text)

    try:
        res = subprocess.run(
            ["sbatch", str(sbatch_path)],
            capture_output=True, text=True, check=True,
        )
        submitted.append((log_stem, res.stdout.strip()))
        print(f"submitted: {log_stem}  →  {res.stdout.strip()}")
    except subprocess.CalledProcessError as exc:
        failed.append((log_stem, exc.stderr.strip()))
        print(f"FAILED SUBMISSION: {log_stem}")
        print(exc.stderr)

print(f"\nsubmitted={len(submitted)}, failed={len(failed)}")

Submitting 18 jobs…
submitted: YFC_23512908  →  Submitted batch job 477172
submitted: YFG_18486634  →  Submitted batch job 477173
submitted: YFG_18486638  →  Submitted batch job 477174
submitted: YFG_18486644  →  Submitted batch job 477175
submitted: YFG_23505577  →  Submitted batch job 477176
submitted: YFG_23512012  →  Submitted batch job 477177
submitted: YFG_23512014  →  Submitted batch job 477178
submitted: YFG_23512906  →  Submitted batch job 477179
submitted: YFG_23512908  →  Submitted batch job 477180
submitted: YFK_18486644  →  Submitted batch job 477181
submitted: YFV_18486638  →  Submitted batch job 477182
submitted: YFV_18486644  →  Submitted batch job 477183
submitted: YFV_23505577  →  Submitted batch job 477184
submitted: YFV_23512012  →  Submitted batch job 477185
submitted: YFV_23512013  →  Submitted batch job 477186
submitted: YFV_23512014  →  Submitted batch job 477187
submitted: YFV_23512906  →  Submitted batch job 477188
submitted: YFV_23512908  →  Submitted batch j

## 3 — Re-scan status (run after jobs finish)

In [4]:
camera_rows = discover_cameras()
status_df = pd.DataFrame(camera_rows)

n_pts = status_df["patient"].nunique() if not status_df.empty else 0
print(f"Patients: {n_pts}")
print(f"Camera jobs: {len(status_df)}")
if not status_df.empty:
    print(f"  has synced MP4: {status_df['has_mp4'].sum()}")
    print(f"  pending:        {(status_df['ready'] & ~status_df['has_mp4']).sum()}")

status_df[["patient", "task_keyword", "cam_serial", "has_mp4", "submit"]]

Patients: 12
Camera jobs: 89
  has synced MP4: 79
  pending:        10


,patient,task_keyword,cam_serial,has_mp4,submit
0,YFC,EMU-0028_convo,18486634,True,False
1,YFC,EMU-0028_convo,18486638,True,False
2,YFC,EMU-0028_convo,23512014,True,False
3,YFC,EMU-0028_convo,23512906,True,False
4,YFC,EMU-0028_convo,23512908,False,True
...,...,...,...,...,...
84,YFV,EMU-0208_convo,23512012,True,False
85,YFV,EMU-0208_convo,23512013,True,False
86,YFV,EMU-0208_convo,23512014,True,False
87,YFV,EMU-0208_convo,23512906,True,False


## 4 — Inspect missing cameras (run after jobs finish)

Per-camera errors are captured in the SLURM `.err` logs under `asd_slurm_logs/`.
This cell lists any cameras that are still missing a synced MP4 despite being ready.

In [5]:
missing = status_df[status_df["ready"] & ~status_df["has_mp4"]] if not status_df.empty else pd.DataFrame()
print(f"Cameras missing synced MP4: {len(missing)}")

for _, row in missing.iterrows():
    print(f"  {row['patient']}  cam={row['cam_serial']}  task={row['task_keyword']}")
    print(f"    expected: {row['synced_dir']}")
    # Check SLURM err log (most recent for this patient+camera)
    log_pattern = f"{row['patient']}_{row['cam_serial']}_*.err"
    err_logs = sorted(SLURM_LOGS_ROOT.glob(log_pattern))
    if err_logs:
        print(f"    SLURM err: {err_logs[-1]}")
        print(err_logs[-1].read_text(errors="ignore")[-2000:])

Cameras missing synced MP4: 10
  YFC  cam=23512908  task=EMU-0028_convo
    expected: /mnt/labworlds/Hayden/Hayden_Lab/speech_247/anilu_comparison/YFC/video/YFC/EMU-0028_convo/23512908/synced_video
    SLURM err: /mnt/labworlds/Hayden/Hayden_Lab/convo_comparison/asd_slurm_logs/YFC_23512908_477172.err
FAILED: YFC
Traceback (most recent call last):
  File "/scratch/tahaismail424/speech_247/notebooks/REBIRTH_2!/convo_comparison/convo_asd_worker.py", line 102, in main
    raise RuntimeError(
RuntimeError: cli_emu_time exited with code 4. See /mnt/labworlds/Hayden/Hayden_Lab/speech_247/anilu_comparison/YFC/video/asd_stderr.log


  YFG  cam=18486634  task=EMU-0018_convo
    expected: /mnt/labworlds/Hayden/Hayden_Lab/speech_247/anilu_comparison/YFG/video/YFG/EMU-0018_convo/18486634/synced_video
    SLURM err: /mnt/labworlds/Hayden/Hayden_Lab/convo_comparison/asd_slurm_logs/YFG_18486634_477173.err
FAILED: YFG
Traceback (most recent call last):
  File "/scratch/tahaismail424/speech_247/notebook

## 5 — Inspect outputs

Lists synced MP4s produced for each successful patient.

In [6]:
success_rows = status_df[status_df["has_mp4"]] if not status_df.empty else pd.DataFrame()
print(f"Cameras with synced MP4: {len(success_rows)}")

for _, row in success_rows.iterrows():
    synced_dir = row["synced_dir"]
    mp4s = sorted(synced_dir.glob("*.mp4")) if synced_dir and synced_dir.exists() else []
    total_mb = sum(m.stat().st_size for m in mp4s) / 1e6
    print(f"\n{row['patient']}  cam={row['cam_serial']}  task={row['task_keyword']}")
    for mp4 in mp4s:
        size_mb = mp4.stat().st_size / 1e6
        print(f"  {mp4.name}  ({size_mb:.1f} MB)")
    if len(mp4s) > 1:
        print(f"  total: {total_mb:.1f} MB")

Cameras with synced MP4: 79

YFC  cam=18486634  task=EMU-0028_convo
  EMU-0028_convo_18486634.mp4  (584.3 MB)

YFC  cam=18486638  task=EMU-0028_convo
  EMU-0028_convo_18486638.mp4  (376.5 MB)

YFC  cam=23512014  task=EMU-0028_convo
  EMU-0028_convo_23512014.mp4  (306.6 MB)

YFC  cam=23512906  task=EMU-0028_convo
  EMU-0028_convo_23512906.mp4  (386.8 MB)

YFF  cam=18486634  task=EMU-0017_convo
  EMU-0017_convo_18486634.mp4  (682.0 MB)

YFF  cam=18486638  task=EMU-0017_convo
  EMU-0017_convo_18486638.mp4  (429.6 MB)

YFF  cam=23512014  task=EMU-0017_convo
  EMU-0017_convo_23512014.mp4  (383.9 MB)

YFF  cam=23512906  task=EMU-0017_convo
  EMU-0017_convo_23512906.mp4  (548.4 MB)

YFI  cam=18486634  task=EMU-0081_convo
  EMU-0081_convo_18486634.mp4  (457.5 MB)

YFI  cam=18486638  task=EMU-0081_convo
  EMU-0081_convo_18486638.mp4  (395.8 MB)

YFI  cam=18486644  task=EMU-0081_convo
  EMU-0081_convo_18486644.mp4  (325.0 MB)

YFI  cam=23505577  task=EMU-0081_convo
  EMU-0081_convo_23505577.mp4 

## 6 — Active Speaker Detection (LR-ASD)

Runs `LR-ASD` (`Columbia_test.py`) on every synced MP4 produced by the video-sync step.
One SLURM GPU job is submitted **per camera MP4** so all cameras can run in parallel.
You can later pick the best-resolving camera for downstream analysis.

Worker reused from vad_new pipeline: `lrasd_interval_worker.py`


In [7]:
# ── LR-ASD paths ────────────────────────────────────────────────────────────
LRASD_REPO_ROOT  = Path('/scratch/tahaismail424/asd_testing/LR-ASD')
LRASD_MODEL_PATH = LRASD_REPO_ROOT / 'weight' / 'finetuning_TalkSet.model'
LRASD_WORKER_PATH = Path('/scratch/tahaismail424/speech_247/notebooks/REBIRTH_2!/video_processing/lrasd_interval_worker.py')
ASD_PYTHON_BIN   = '/scratch/tahaismail424/miniforge3/envs/asd_testing/bin/python3'
ASD_CONDA_ACTIVATE = 'source /scratch/tahaismail424/.bash_profile && conda activate asd_testing'

# ── LR-ASD SLURM settings ───────────────────────────────────────────────────
ASD_GRES      = 'mps:l40:16'
ASD_CPUS      = 8
ASD_MEM       = '64G'
ASD_TIME      = '24:00:00'
ASD_QOS       = 'big_batch_tier'

# Set True to resubmit even cameras that already have _SUCCESS
FORCE_RESUBMIT_ASD = False

ASD_LOGS_ROOT    = ANILU_ROOT.parent.parent / 'convo_comparison' / 'lrasd_slurm_logs'
ASD_SCRIPTS_ROOT = ANILU_ROOT.parent.parent / 'convo_comparison' / 'lrasd_slurm_scripts'
ASD_LOGS_ROOT.mkdir(parents=True, exist_ok=True)
ASD_SCRIPTS_ROOT.mkdir(parents=True, exist_ok=True)

print('LRASD_REPO_ROOT: ', LRASD_REPO_ROOT)
print('LRASD_MODEL:     ', LRASD_MODEL_PATH)
print('LRASD_WORKER:    ', LRASD_WORKER_PATH)
print('ASD logs:        ', ASD_LOGS_ROOT)
print('model exists:    ', LRASD_MODEL_PATH.exists())
print('worker exists:   ', LRASD_WORKER_PATH.exists())


LRASD_REPO_ROOT:  /scratch/tahaismail424/asd_testing/LR-ASD
LRASD_MODEL:      /scratch/tahaismail424/asd_testing/LR-ASD/weight/finetuning_TalkSet.model
LRASD_WORKER:     /scratch/tahaismail424/speech_247/notebooks/REBIRTH_2!/video_processing/lrasd_interval_worker.py
ASD logs:         /mnt/labworlds/Hayden/Hayden_Lab/convo_comparison/lrasd_slurm_logs
model exists:     True
worker exists:    True


## 7 — Scan synced MP4s & ASD status

In [8]:
def derive_asd_status(mp4_path: Path) -> dict:
    """Derive LR-ASD output paths and check status markers for one synced MP4."""
    # Synced MP4 layout produced by cli_emu_time:
    #   anilu_comparison/{pt}/video/{pt}/{task_id}/{cam_serial}/synced_video/{stem}.mp4
    synced_video_dir = mp4_path.parent        # .../synced_video/
    cam_serial       = synced_video_dir.parent.name
    task_id          = synced_video_dir.parent.parent.name
    pt               = synced_video_dir.parent.parent.parent.name

    # LR-ASD outputs under synced_video/{stem}/
    output_dir   = synced_video_dir / mp4_path.stem
    pyavi_dir    = output_dir / 'pyavi'
    success_path = pyavi_dir / '_SUCCESS'
    error_path   = synced_video_dir / f'{mp4_path.stem}.asd_error.txt'
    video_out    = pyavi_dir / 'video_out.avi'

    return {
        'patient':     pt,
        'task_id':     task_id,
        'cam_serial':  cam_serial,
        'mp4_path':    mp4_path,
        'output_dir':  output_dir,
        'pyavi_dir':   pyavi_dir,
        'success_path': success_path,
        'error_path':  error_path,
        'video_out':   video_out,
        'has_success': success_path.exists(),
        'has_error':   error_path.exists(),
        'has_video_out': video_out.exists(),
        'ready_for_asd': (
            (not success_path.exists() or FORCE_RESUBMIT_ASD)
        ),
    }


# Glob all synced MP4s written by cli_emu_time across all patients.
# Pattern: anilu_comparison/{pt}/video/{pt}/{task_id}/{cam_serial}/synced_video/*.mp4
all_mp4s = sorted(ANILU_ROOT.glob('*/video/*/*/*/synced_video/*.mp4'))
asd_rows = [derive_asd_status(p) for p in all_mp4s]
asd_df   = pd.DataFrame(asd_rows)

print(f'Synced MP4s found: {len(asd_df)}')
if not asd_df.empty:
    print(f'  has _SUCCESS:   {asd_df["has_success"].sum()}')
    print(f'  has error:      {asd_df["has_error"].sum()}')
    print(f'  has video_out:  {asd_df["has_video_out"].sum()}')
    print(f'  ready_for_asd:  {asd_df["ready_for_asd"].sum()}')

asd_df[['patient','task_id','cam_serial','has_success','has_error','has_video_out','ready_for_asd','mp4_path']] if not asd_df.empty else pd.DataFrame()


Synced MP4s found: 79
  has _SUCCESS:   57
  has error:      14
  has video_out:  57
  ready_for_asd:  22


,patient,task_id,cam_serial,has_success,has_error,has_video_out,ready_for_asd,mp4_path
0,YFC,EMU-0028_convo,18486634,True,False,True,False,/mnt/labworlds/Hayden/Hayden_Lab/speech_247/an...
1,YFC,EMU-0028_convo,18486638,True,False,True,False,/mnt/labworlds/Hayden/Hayden_Lab/speech_247/an...
2,YFC,EMU-0028_convo,23512014,True,False,True,False,/mnt/labworlds/Hayden/Hayden_Lab/speech_247/an...
3,YFC,EMU-0028_convo,23512906,True,False,True,False,/mnt/labworlds/Hayden/Hayden_Lab/speech_247/an...
4,YFF,EMU-0017_convo,18486634,True,False,True,False,/mnt/labworlds/Hayden/Hayden_Lab/speech_247/an...
...,...,...,...,...,...,...,...,...
74,YFV,EMU-0208_convo,23512012,False,False,False,True,/mnt/labworlds/Hayden/Hayden_Lab/speech_247/an...
75,YFV,EMU-0208_convo,23512013,False,False,False,True,/mnt/labworlds/Hayden/Hayden_Lab/speech_247/an...
76,YFV,EMU-0208_convo,23512014,False,False,False,True,/mnt/labworlds/Hayden/Hayden_Lab/speech_247/an...
77,YFV,EMU-0208_convo,23512906,False,False,False,True,/mnt/labworlds/Hayden/Hayden_Lab/speech_247/an...


## 8 — Submit LR-ASD SLURM jobs (one per camera MP4)

In [9]:
asd_submitted = []
asd_failed    = []

to_run = asd_df[asd_df['ready_for_asd']] if not asd_df.empty else pd.DataFrame()
print(f'Submitting {len(to_run)} ASD jobs…')

for _, row in to_run.iterrows():
    mp4_path    = row['mp4_path']
    pt          = row['patient']
    cam_serial  = row['cam_serial']
    task_id     = row['task_id']
    job_name    = f'lrasd_{pt}_{cam_serial}'
    log_stem    = f'{pt}_{task_id}_{cam_serial}'

    reset_line = (
        f'rm -f {row["success_path"]} {row["error_path"]}'
        if FORCE_RESUBMIT_ASD else 'true'
    )

    lines = [
        '#!/bin/bash',
        f'#SBATCH --job-name={job_name}',
        f'#SBATCH --partition={PARTITION}',
        f'#SBATCH --gres={ASD_GRES}',
        '#SBATCH --exclude=guppy-1',
        f'#SBATCH --cpus-per-task={ASD_CPUS}',
        f'#SBATCH --mem={ASD_MEM}',
        f'#SBATCH --time={ASD_TIME}',
        f'#SBATCH --qos={ASD_QOS}',
        f'#SBATCH --output={ASD_LOGS_ROOT}/{log_stem}_%j.out',
        f'#SBATCH --error={ASD_LOGS_ROOT}/{log_stem}_%j.err',
        '',
        'set -eo pipefail',
        '',
        ASD_CONDA_ACTIVATE,
        '',
        'echo "HOSTNAME: $(hostname)"',
        'echo "START: $(date)"',
        f'echo "PATIENT:    {pt}"',
        f'echo "TASK_ID:    {task_id}"',
        f'echo "CAM_SERIAL: {cam_serial}"',
        f'echo "MP4:        {mp4_path}"',
        '',
        reset_line,
        '',
        f'{ASD_PYTHON_BIN} {LRASD_WORKER_PATH} {mp4_path} {LRASD_REPO_ROOT} {LRASD_MODEL_PATH}',
        '',
        'echo "END: $(date)"',
    ]
    sbatch_text = '\n'.join(lines) + '\n'
    sbatch_path = ASD_SCRIPTS_ROOT / f'{log_stem}.sbatch'
    sbatch_path.write_text(sbatch_text)

    try:
        res = subprocess.run(
            ['sbatch', str(sbatch_path)],
            capture_output=True, text=True, check=True,
        )
        asd_submitted.append((log_stem, res.stdout.strip()))
        print(f'submitted: {log_stem}  →  {res.stdout.strip()}')
    except subprocess.CalledProcessError as exc:
        asd_failed.append((log_stem, exc.stderr.strip()))
        print(f'FAILED: {log_stem}')
        print(exc.stderr)

print(f'\nasd_submitted={len(asd_submitted)}, asd_failed={len(asd_failed)}')


Submitting 22 ASD jobs…
submitted: YFK_EMU-0040_convo_23512013  →  Submitted batch job 477190
submitted: YFP_EMU-0088_convo_18486638  →  Submitted batch job 477191
submitted: YFP_EMU-0088_convo_18486644  →  Submitted batch job 477192
submitted: YFP_EMU-0088_convo_23505577  →  Submitted batch job 477193
submitted: YFP_EMU-0088_convo_23512012  →  Submitted batch job 477194
submitted: YFR_EMU-0091_convo_23512012  →  Submitted batch job 477195
submitted: YFR_EMU-0091_convo_23512013  →  Submitted batch job 477196
submitted: YFR_EMU-0091_convo_23512014  →  Submitted batch job 477197
submitted: YFR_EMU-0091_convo_23512906  →  Submitted batch job 477198
submitted: YFR_EMU-0091_convo_23512908  →  Submitted batch job 477199
submitted: YFS_EMU-0089_convo_18486644  →  Submitted batch job 477200
submitted: YFS_EMU-0089_convo_23512012  →  Submitted batch job 477201
submitted: YFS_EMU-0089_convo_23512014  →  Submitted batch job 477202
submitted: YFU_EMU-0224_convo_23512906  →  Submitted batch job 477

## 9 — Re-scan ASD status (run after jobs finish)

In [7]:
all_mp4s = sorted(ANILU_ROOT.glob('*/video/*/*/*/synced_video/*.mp4'))
asd_rows = [derive_asd_status(p) for p in all_mp4s]
asd_df   = pd.DataFrame(asd_rows)

print(f'Synced MP4s: {len(asd_df)}')
if not asd_df.empty:
    print(f'  _SUCCESS:   {asd_df["has_success"].sum()}')
    print(f'  errors:     {asd_df["has_error"].sum()}')
    print(f'  video_out:  {asd_df["has_video_out"].sum()}')
    print(f'  pending:    {asd_df["ready_for_asd"].sum()}')

asd_df[['patient','task_id','cam_serial','has_success','has_error','has_video_out']] if not asd_df.empty else pd.DataFrame()


Synced MP4s: 30
  _SUCCESS:   30
  errors:     0
  video_out:  30
  pending:    0


,patient,task_id,cam_serial,has_success,has_error,has_video_out
0,YFC,EMU-0028_convo,18486634,True,False,True
1,YFC,EMU-0028_convo,18486638,True,False,True
2,YFC,EMU-0028_convo,23512014,True,False,True
3,YFC,EMU-0028_convo,23512906,True,False,True
4,YFF,EMU-0017_convo,18486634,True,False,True
5,YFF,EMU-0017_convo,18486638,True,False,True
6,YFF,EMU-0017_convo,23512014,True,False,True
7,YFF,EMU-0017_convo,23512906,True,False,True
8,YFI,EMU-0081_convo,18486634,True,False,True
9,YFI,EMU-0081_convo,18486638,True,False,True


## 10 — Inspect ASD errors

In [8]:
error_rows = asd_df[asd_df['has_error']] if not asd_df.empty else pd.DataFrame()
print(f'MP4s with ASD error: {len(error_rows)}')

for _, row in error_rows.iterrows():
    print('=' * 100)
    print(f"Patient: {row['patient']}  task: {row['task_id']}  cam: {row['cam_serial']}")
    print(f"  mp4: {row['mp4_path']}")
    if row['error_path'].exists():
        print(row['error_path'].read_text()[:5000])


MP4s with ASD error: 0


## 11 — Inspect ASD outputs

Lists successful ASD runs grouped by patient and camera.

In [9]:
success_rows = asd_df[asd_df['has_success']] if not asd_df.empty else pd.DataFrame()
print(f'Successful ASD runs: {len(success_rows)}')

for _, row in success_rows.iterrows():
    pyavi_dir  = row['pyavi_dir']
    video_out  = row['video_out']
    size_mb    = video_out.stat().st_size / 1e6 if video_out.exists() else 0
    csv_files  = sorted(pyavi_dir.parent.rglob('*.csv')) if pyavi_dir.parent.exists() else []
    print(f"\n{row['patient']}  cam={row['cam_serial']}  task={row['task_id']}")
    print(f"  video_out.avi: {size_mb:.1f} MB")
    for csv in csv_files:
        print(f"  csv: {csv.relative_to(row['output_dir'])}")


Successful ASD runs: 30

YFC  cam=18486634  task=EMU-0028_convo
  video_out.avi: 869.9 MB

YFC  cam=18486638  task=EMU-0028_convo
  video_out.avi: 496.6 MB

YFC  cam=23512014  task=EMU-0028_convo
  video_out.avi: 426.6 MB

YFC  cam=23512906  task=EMU-0028_convo
  video_out.avi: 592.0 MB

YFF  cam=18486634  task=EMU-0017_convo
  video_out.avi: 912.8 MB

YFF  cam=18486638  task=EMU-0017_convo
  video_out.avi: 534.2 MB

YFF  cam=23512014  task=EMU-0017_convo
  video_out.avi: 530.8 MB

YFF  cam=23512906  task=EMU-0017_convo
  video_out.avi: 779.4 MB

YFI  cam=18486634  task=EMU-0081_convo
  video_out.avi: 784.3 MB

YFI  cam=18486638  task=EMU-0081_convo
  video_out.avi: 583.8 MB

YFI  cam=18486644  task=EMU-0081_convo
  video_out.avi: 518.6 MB

YFI  cam=23505577  task=EMU-0081_convo
  video_out.avi: 429.9 MB

YFI  cam=23512012  task=EMU-0081_convo
  video_out.avi: 346.0 MB

YFI  cam=23512014  task=EMU-0081_convo
  video_out.avi: 478.1 MB

YFI  cam=23512906  task=EMU-0081_convo
  video_out.